In [60]:
import pandas as pd
df = pd.read_csv("../data/raw/2026-10-22.csv")
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 126 entries, 0 to 125
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   id      126 non-null    int64  
 1   name    126 non-null    str    
 2   price   126 non-null    float64
 3   date    126 non-null    str    
dtypes: float64(1), int64(1), str(2)
memory usage: 4.1 KB


In [61]:
df.head()


,id,name,price,date
0,4753925,MSI VGA GeForce RTX 3050 VENTUS 2X E OC 6 GB,269.0,2026-10-22
1,4796691,PNY VGA Nvidia RTX PRO 6000 Blackwell 96 GB,19999.0,2026-10-22
2,5339936,Asus VGA GeForce RTX 5090 ROG Astral Edition 2...,7999.0,2026-10-22
3,4791908,MSI VGA GeForce RTX RTX 5060 SHADOW 2X OC 8 GB,429.0,2026-10-22
4,4810066,Sapphire VGA Radeon RX 9060 XT NITRO+ OC 16 GB,759.0,2026-10-22


In [62]:
df["name"] = df["name"].str.replace("VGA ", "")
df["brand"] = df["name"].str.split().str[0]


In [63]:
print(df["brand"])

0           MSI
1           PNY
2          Asus
3           MSI
4      Sapphire
         ...   
121    Sapphire
122        Asus
123        Asus
124        Asus
125        Asus
Name: brand, Length: 126, dtype: object


In [64]:
df["brand"].value_counts()

brand
Gigabyte    38
Asus        24
MSI         19
PNY         19
Sapphire    13
Asrock       6
AMD          5
Intel        1
XFX          1
Name: count, dtype: int64

In [65]:
df[df["brand"] == "VGA"]

,id,name,price,date,brand


In [66]:
df["name"].str.contains("Radeon|AMD", case = False).sum()

np.int64(35)

In [67]:
df["name"].str.contains("GeForce|Nvidia", case = False).sum()

np.int64(83)

In [68]:
df["name"].str.contains("Intel|Arc", case = False).sum()

np.int64(3)

In [69]:
amd = df["name"].str.contains("Radeon|AMD", case = False)
nvidia = df["name"].str.contains("GeForce|Nvidia", case = False)
intel = df["name"].str.contains("Intel|Arc", case = False)
df[~(amd|nvidia|intel)]

,id,name,price,date,brand
64,4590023,PNY Quadro A400 Low Profile 4 GB,349.0,2026-10-22,PNY
99,4590031,PNY Quadro A1000 Low Profile 8 GB,799.0,2026-10-22,PNY
115,4042328,PNY Quadro T400 Low Profile 4 GB,199.0,2026-10-22,PNY
117,3494659,PNY Quadro P620 V2 2 GB,199.9,2026-10-22,PNY
118,2902818,PNY Quadro P620 2GB GDDR5,249.9,2026-10-22,PNY


In [70]:
df["name"].str.contains("GeForce|Nvidia|PNY", case = False).sum()
nvidia = df["name"].str.contains("GeForce|Nvidia|PNY", case = False)

In [71]:
df[~(amd|nvidia|intel)]

,id,name,price,date,brand


In [72]:
amd.sum() + nvidia.sum() + intel.sum()

np.int64(126)

In [73]:
df.loc[amd, "gpu_maker"] = "AMD"
df.loc[nvidia, "gpu_maker"] = "NVIDIA"
df.loc[intel, "gpu_maker"] = "INTEL"

In [74]:
df["gpu_maker"].value_counts()

gpu_maker
NVIDIA    88
AMD       35
INTEL      3
Name: count, dtype: int64

In [75]:
df.head(10)

,id,name,price,date,brand,gpu_maker
0,4753925,MSI GeForce RTX 3050 VENTUS 2X E OC 6 GB,269.0,2026-10-22,MSI,NVIDIA
1,4796691,PNY Nvidia RTX PRO 6000 Blackwell 96 GB,19999.0,2026-10-22,PNY,NVIDIA
2,5339936,Asus GeForce RTX 5090 ROG Astral Edition 20 32 GB,7999.0,2026-10-22,Asus,NVIDIA
3,4791908,MSI GeForce RTX RTX 5060 SHADOW 2X OC 8 GB,429.0,2026-10-22,MSI,NVIDIA
4,4810066,Sapphire Radeon RX 9060 XT NITRO+ OC 16 GB,759.0,2026-10-22,Sapphire,AMD
5,4810090,Sapphire Radeon RX 9060 XT PULSE OC 16 GB,719.0,2026-10-22,Sapphire,AMD
6,4699440,Sapphire Radeon RX 9070 XT NITRO+ OC 16 GB,1199.0,2026-10-22,Sapphire,AMD
7,4820118,Gigabyte GeForce GT 710D3 2 GB,69.9,2026-10-22,Gigabyte,NVIDIA
8,5337283,Asus GeForce RTX 5060 Dual OC Advanced 8 GB,599.0,2026-10-22,Asus,NVIDIA
9,5164052,Asus Radeon RX 9070 GRE PRIME OC 12 GB,749.0,2026-10-22,Asus,AMD


In [76]:
amd_pro_cards= df["name"].str.contains("Radeon pro", case = False)
nvidia_pro_cards = df["name"].str.contains("Quadro|RTX PRO", case = False)
intel_pro_cards = df["name"].str.contains("Arc Pro", case = False)


In [77]:
df["is_pro"] = (amd_pro_cards|nvidia_pro_cards|intel_pro_cards)

## Separating professional and gaming cards

**Separation is based on the manufacturers' professional product lines: Quadro and RTX PRO (NVIDIA), Radeon Pro (AMD) and Arc Pro (Intel). Use does not matter: a gaming card such as the RTX 5080 still counts as gaming, even if it is used for professional work.**

In [78]:
df["is_pro"].value_counts()

is_pro
False    106
True      20
Name: count, dtype: int64

In [79]:
df["vram"] = df["name"].str.extract(r"(\d+)\s?GB?\b").astype(float)

In [80]:
df["vram"].value_counts()

vram
16.0    39
8.0     31
12.0    17
32.0    13
6.0      4
2.0      4
4.0      4
48.0     4
24.0     3
96.0     2
20.0     2
10.0     1
Name: count, dtype: int64

In [81]:
df[df["vram"].isna()]

,id,name,price,date,brand,gpu_maker,is_pro,vram
12,4761812,MSI GeForce RTX 5080 Ventus 3X White OC,1799.0,2026-10-22,MSI,NVIDIA,False,NaN
103,4500148,PNY NVIDIA RTX PRO SYNC,999.0,2026-10-22,PNY,NVIDIA,True,NaN


In [82]:
is_5080 = df["name"].str.contains("5080")

In [83]:
df[is_5080]

,id,name,price,date,brand,gpu_maker,is_pro,vram
12,4761812,MSI GeForce RTX 5080 Ventus 3X White OC,1799.0,2026-10-22,MSI,NVIDIA,False,NaN
21,4706048,Gigabyte GeForce RTX 5080 AORUS MASTER 16 GB,2099.0,2026-10-22,Gigabyte,NVIDIA,False,16.0
24,4688619,Asus GeForce RTX 5080 TUF GAMING OC 16 GB,1999.0,2026-10-22,Asus,NVIDIA,False,16.0
51,4706056,Gigabyte GeForce RTX 5080 GAMING OC 16 GB,1999.0,2026-10-22,Gigabyte,NVIDIA,False,16.0
52,4706064,Gigabyte GeForce RTX 5080 AERO OC SFF 16 GB,2299.0,2026-10-22,Gigabyte,NVIDIA,False,16.0
54,5340772,Gigabyte GeForce RTX 5080 AORUS INFINITY 16 GB,2499.0,2026-10-22,Gigabyte,NVIDIA,False,16.0
55,4706072,Gigabyte GeForce RTX 5080 WINDFORCE OC SFF 16 GB,1949.0,2026-10-22,Gigabyte,NVIDIA,False,16.0
125,4866029,Asus GeForce RTX 5080 ROG ASTRAL GAMING OC WHI...,2249.0,2026-10-22,Asus,NVIDIA,False,16.0


In [84]:
df[df["vram"].isna()]

,id,name,price,date,brand,gpu_maker,is_pro,vram
12,4761812,MSI GeForce RTX 5080 Ventus 3X White OC,1799.0,2026-10-22,MSI,NVIDIA,False,NaN
103,4500148,PNY NVIDIA RTX PRO SYNC,999.0,2026-10-22,PNY,NVIDIA,True,NaN


In [86]:
df["vram"].value_counts()

vram
16.0    39
8.0     31
12.0    17
32.0    13
6.0      4
2.0      4
4.0      4
48.0     4
24.0     3
96.0     2
20.0     2
10.0     1
Name: count, dtype: int64

## Filling in missing VRAM

Two rows have no VRAM after the extraction:

- **MSI GeForce RTX 5080 Ventus 3X White OC** (id 4761812): the product name does not include the VRAM. Every other RTX 5080 in the data has 16 GB, so we set it to 16 manually.
- **PNY NVIDIA RTX PRO SYNC** (id 4500148): this is a display sync accessory, not a graphics card, so it has no VRAM and stays empty.

In [87]:
is_id_4761812 = df["id"] == 4761812
df.loc[is_id_4761812, "vram"] = 16

In [88]:
df[df["vram"].isna()]

,id,name,price,date,brand,gpu_maker,is_pro,vram
103,4500148,PNY NVIDIA RTX PRO SYNC,999.0,2026-10-22,PNY,NVIDIA,True,NaN
